# Plot saved results
Set `root` to the folder containing your experiment results. Paths are relative
to the notebook's working directory. The cells below read saved scalar aggregate
metrics directly from HDF5; they do not recompute means, medians, or other scores.

Leave `csv_path` as `None` to work without a CSV, or set it to an output path and
run the final cell to export the table. See the [README](../README.md#export-and-plot-results)
for the metric keys.


In [ ]:
from pathlib import Path
import h5py
import pandas as pd
from rich import print

root = Path("../outputs/paper")
csv_path = None  # For example: Path("../results/paper_summary.csv")


In [ ]:
if not root.is_dir():
  raise FileNotFoundError(f"Results directory does not exist: {root}")

rows = []
for path in sorted(root.rglob("*.hdf5")):
  with h5py.File(path, "r") as results:
    if "aggregate" not in results:
      continue
    for statistic in results["aggregate"]:
      value = results["aggregate"][statistic][()]
      if getattr(value, "ndim", 0) == 0:
        rows.append([str(path.parent.relative_to(root)),
                     path.stem, statistic, float(value)])

scores = pd.DataFrame(rows, columns=["run", "metric", "statistic", "value"])
scores


## Plot a statistic
Choose a saved statistic from the table above.


In [ ]:
statistic = "linv_conv"
view = scores.loc[scores.statistic.eq(statistic), ["run", "value"]]
if view.empty:
  print(f"No saved values for {statistic}.")
else:
  view.set_index("run").plot.barh(title=statistic, legend=False)


## Optional CSV export
Set `csv_path` in the first code cell to save the table. Exporting replaces an
existing CSV at that path.


In [ ]:
if csv_path is not None:
  csv_path.parent.mkdir(parents=True, exist_ok=True)
  scores.to_csv(csv_path, index=False, na_rep="nan")
